In [1]:
import ir_datasets
from topic_gen.models import TRECTopic, Topics
import json
import pandas as pd
import os

import sys
sys.path.append("../query_sim_validation")


In [2]:
dataset = ir_datasets.load("disks45/nocr/trec-robust-2004")

In [3]:
generated_topics = []
with open("../data/processed/results/topics-robust-QwenQwen3-30B-A3B-Instruct-2507-FP8-trec-base-q1-d1.jsonl", "r") as f:
    for line in f:
        topic = json.loads(line)
        generated_topics.append(topic)


In [4]:
from query_sim_validation.query_sim_validation.validator import Validator
from query_sim_validation.query_sim_validation.models import Session, Interaction

/home/vscode/.cache/pypoetry/virtualenvs/src-B2WAz2j2-py3.11/lib/python3.11/site-packages/lexical_diversity/lex_div.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [5]:
original = []
for topic in dataset.queries_iter():
    interactions = Interaction(q=topic.title, serp=[], clicks=[])
    session = Session(id=topic.query_id, sid=topic.query_id, interactions=[interactions])
    original.append(session)

In [6]:
simulated = []
for topic in generated_topics:
    interactions = Interaction(q=topic["title"], serp=[], clicks=[])
    session = Session(id=topic["qid"], sid=topic["qid"], interactions=[interactions])
    simulated.append(session)

In [7]:
validator = Validator(original_sessions=original[200:210], simulated_sessions=simulated)

In [8]:
def validate(simulated, original, measures):
    validator = Validator(original_sessions=original, simulated_sessions=simulated)
    results = {}
    for measure in measures:
        res = validator.validate_simulation(measure)
        if "comparison_measures" in res.keys():
            results[measure] = res["comparison_measures"]
    return results

In [9]:
data = validate(simulated, original[200:210], measures=["jaccard_similarity", "cosine_similarity", "wordnet_similarity", "bert_score", "query_length", "query_num_terms", "query_num_named_entities", "flesch_kincaid_scores"])

In [10]:
df = pd.DataFrame(data)

In [11]:
df.mean()

jaccard_similarity    0.405000
cosine_similarity     0.877962
wordnet_similarity    0.245556
bert_score            0.707485
dtype: float64

In [12]:
validator.validate_simulation("query_length")

{'original_measures': [22, 17, 20, 16, 23, 23, 20, 17, 20, 25],
 'simulated_measures': [26, 17, 20, 29, 39, 42, 24, 30, 30, 28]}

In [13]:
validator.validate_simulation("query_num_terms")

{'original_measures': [3, 3, 3, 2, 3, 3, 3, 2, 3, 3],
 'simulated_measures': [3, 3, 3, 3, 6, 5, 4, 4, 4, 4]}

In [14]:
validator.validate_simulation("query_num_named_entities")

{'original_measures': [1, 2, 1, 0, 0, 0, 0, 0, 1, 1],
 'simulated_measures': [0, 2, 1, 1, 0, 1, 0, 0, 0, 1]}

In [15]:
validator.validate_simulation("flesch_kincaid_scores")

{'original_measures': [13.113333333333333,
  1.3133333333333361,
  9.180000000000003,
  2.890000000000004,
  13.113333333333333,
  9.180000000000003,
  -2.619999999999999,
  14.690000000000001,
  1.3133333333333361,
  13.113333333333333],
 'simulated_measures': [13.113333333333333,
  1.3133333333333361,
  9.180000000000003,
  28.846666666666668,
  8.383333333333333,
  12.320000000000004,
  3.6700000000000017,
  9.57,
  9.57,
  9.57]}

# Matrix

In [18]:
def parse_topic(generated_topics, component):
    objs = []
    for topic in generated_topics:
        interactions = Interaction(q=topic[component], serp=[], clicks=[])
        session = Session(id=topic["qid"], sid=topic["qid"], interactions=[interactions])
        objs.append(session)
    return objs

def load_topics(topic_path):
    raw_topics = []
    with open(topic_path, "r") as f:
        for line in f:
            topic = json.loads(line)
            raw_topics.append(topic)
            
    simulated_title = parse_topic(raw_topics, "title")
    simulated_description = parse_topic(raw_topics, "description")
    simulated_narrative = parse_topic(raw_topics, "narrative")
    return simulated_title, simulated_description, simulated_narrative

def parse_name(name):
    queries, docs = name.strip(".jsonl").split("-")[-2:]
    return queries[1:], docs[1:]

def evaluate_artefact(simulated, original, queries, docs):
    ret = []
    measures=["jaccard_similarity", "cosine_similarity", "wordnet_similarity", "bert_score"]
    for measure in measures:    
        validator = Validator(original_sessions=original, simulated_sessions=simulated)
        res = validator.validate_simulation(measure)["comparison_measures"]
        
    
    for qid, result in enumerate(res):
        ret.append({
            "measure": measure,
            "qid": qid,
            "value": result,
            "nqueries": queries,
            "ndocs": docs,
        })
    return ret

In [19]:
original = []
for topic in dataset.queries_iter():
    interactions = Interaction(q=topic.title, serp=[], clicks=[])
    session = Session(id=topic.query_id, sid=topic.query_id, interactions=[interactions])
    original.append(session)

In [20]:
import time
from tqdm import tqdm 

base_dir = "../data/processed/results/"
results = []
for topic_set in tqdm(os.listdir(base_dir)):
    queries, docs = parse_name(topic_set)

    simulated_title, simulated_description, simulated_narrative = load_topics(os.path.join(base_dir, topic_set))
    
    for component in [simulated_title, simulated_description, simulated_narrative]:
        res = evaluate_artefact(component, original[200:210], queries, docs)
        results.extend(res)

  4%|▎         | 3/81 [00:32<13:53, 10.69s/it]HTTP Error 429 thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/./modules.json
Retrying in 1s [Retry 1/5].
HTTP Error 429 thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/./modules.json
Retrying in 2s [Retry 2/5].
HTTP Error 429 thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/./modules.json
Retrying in 4s [Retry 3/5].
  4%|▎         | 3/81 [00:36<15:39, 12.05s/it]


KeyboardInterrupt: 